# 01 — Ingest and QC

Load Sackmann-style CSVs from `TML_DATA_DIR`, record manifest lineage, and inspect completion flags.
All logic lives in `tml.data` — this notebook only orchestrates calls.

In [ ]:
from pathlib import Path

import pandas as pd

from tml.data.identity import PlayerIdentityMap
from tml.data.ingest import ingest_match_files
from tml.data.modeling_table import build_modeling_table
from tml.shared.config import get_settings

In [ ]:
settings = get_settings()
root = settings.tml_data_dir
print(f"Data root: {root.resolve()}")

In [ ]:
# Adjust globs for the years you want to load.
atp_paths = sorted(root.glob("2024.csv"))
challenger_paths = sorted(root.glob("2024_challenger.csv"))

atp = ingest_match_files(atp_paths, root=root, tour_level="atp")
challenger = ingest_match_files(challenger_paths, root=root, tour_level="challenger") if challenger_paths else None

print("ATP snapshot:", atp.dataset_snapshot_id)
print("Manifest rows:", len(atp.manifest))

In [ ]:
matches = atp.matches if challenger is None else pd.concat([atp.matches, challenger.matches], ignore_index=True)
matches["completion_status"].value_counts()

In [ ]:
identity = PlayerIdentityMap()
modeling = build_modeling_table(matches, identity)
modeling.head()